# Subset dinoflagellate single-cell data by ViHoP taxonomy

Standalone script for manuscript code availability. Loads raw Cell Ranger
matrices from three Alps samples, merges them, and retains cells assigned
homogeneously to **Dinophyceae** (dinoflagellate class) based on ViHoP BLAST
summaries.

Output: raw-count AnnData object (`Alps_Dinoflagellata.h5ad`).

No external project modules are required beyond standard Python packages.

In [1]:
import os
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

In [2]:
def load_cellranger_matrix(path: str, sample_name: str = None) -> ad.AnnData:
    """Load a 10x Cell Ranger matrix directory into AnnData with raw counts in `.X`."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Path does not exist: {path}")
    print(path)
    adata = sc.read_10x_mtx(path, var_names="gene_symbols")
    if sample_name:
        adata.obs["sample"] = sample_name
    adata.layers["counts"] = adata.X.copy()
    return adata


def merge_cellranger_samples(sample_paths: Dict[str, str]) -> ad.AnnData:
    """Load and outer-join multiple Cell Ranger matrices with `{barcode}.{sample}` cell IDs."""
    adatas = [
        load_cellranger_matrix(path, sample_name=sample)
        for sample, path in sample_paths.items()
    ]
    merged = ad.concat(
        adatas,
        join="outer",
        label="sample",
        keys=list(sample_paths.keys()),
        index_unique=None,
    )
    merged.obs_names = [
        f"{bc.replace('-1', '')}.{samp}"
        for bc, samp in zip(merged.obs_names, merged.obs["sample"])
    ]
    if merged.obs_names.duplicated().any():
        merged.obs_names_make_unique()
    return merged


def filter_homogeneous_cells(
    total_cells: Sequence[str],
    annotation_file: str,
    threshold: float = 1.0,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Return all and homogeneous cell annotations for cells in the matrix.

    ViHoP `blast_summary.*.csv` files store `percent` as a fraction in [0, 1],
    not as 0–100. A threshold of 1.0 keeps only cells where a single taxon
    accounts for 100% of mapped reads.
    """
    full_annotation = pd.read_csv(annotation_file)
    subset_annotation = full_annotation[full_annotation["cell"].isin(total_cells)]
    filtered_annotation = (
        subset_annotation[subset_annotation["percent"] >= threshold]
        .sort_values("percent", ascending=False)
        .drop_duplicates("cell")
    )
    return subset_annotation, filtered_annotation


def filter_last_blast(
    total_cells: Sequence[str],
    annotation_file: str,
    threshold_last: float = 0.9,
    threshold_blast: float = 0.9,
    exclude_taxa: Sequence[str] = ("Metazoa",),
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Alternative annotation filter using full ViHoP annotation table.

    Not used in the default workflow below; kept for reference.
    """
    full_annotation = pd.read_csv(annotation_file)
    subset_annotation = full_annotation[full_annotation["cell"].isin(total_cells)]
    filtered_annotation = (
        subset_annotation[
            (subset_annotation["Blast percent alignment"] >= threshold_blast)
            & (subset_annotation["last percent mapped reads"] >= threshold_last)
        ]
        .sort_values("last percent mapped reads", ascending=False)
        .drop_duplicates("cell")
    )
    if exclude_taxa:
        filtered_annotation = filtered_annotation[
            ~filtered_annotation["Blast Annotation"].isin(exclude_taxa)
        ]
    return subset_annotation, filtered_annotation


def get_cell_lists(
    subset_annotation: pd.DataFrame,
    filtered_annotation: pd.DataFrame,
    total_cells: Sequence[str],
    level: str,
    taxon: str,
) -> Tuple[List[str], List[str], List[str], List[str]]:
    """Summarize annotated, missing, and taxon-specific cells."""
    existing_cells = subset_annotation["cell"].unique().tolist()
    print(len(existing_cells), "cells with any annotation")

    missing_cells = [c for c in total_cells if c not in existing_cells]
    print(len(missing_cells), "cells without annotation")

    known_cells = filtered_annotation["cell"].unique().tolist()
    print(len(known_cells), "homogeneous cells")

    if taxon != "All known":
        taxon_cells = (
            filtered_annotation.loc[filtered_annotation[level] == taxon, "cell"]
            .unique()
            .tolist()
        )
        print(len(taxon_cells), "cells of", taxon)
    else:
        taxon_cells = []

    return existing_cells, known_cells, taxon_cells, missing_cells


def subset_by_taxa(
    adata: ad.AnnData,
    subset_annotation: pd.DataFrame,
    filtered_annotation: pd.DataFrame,
    total_cells: Sequence[str],
    level: str = "Class",
    taxon: str = "Dinophyceae",
) -> ad.AnnData:
    """Subset AnnData to cells assigned to the requested taxonomic level."""
    _, known_cells, taxon_cells, _ = get_cell_lists(
        subset_annotation,
        filtered_annotation,
        total_cells,
        level,
        taxon,
    )
    if taxon == "All known":
        valid_cells = [c for c in known_cells if c in adata.obs_names]
    else:
        valid_cells = [c for c in taxon_cells if c in adata.obs_names]
    adata_sub = adata[valid_cells, :].copy()
    sc.pp.filter_genes(adata_sub, min_counts=2)
    sc.pp.filter_cells(adata_sub, min_counts=1)
    return adata_sub

In [3]:
# --- configuration ---
base_path = Path("/home/labs/vardi/amirz/pairing/Alps_All_MetaG")

cellranger_paths = {
    "Alps2": base_path / "ViHoP/Alps2/outs/filtered_feature_bc_matrix",
    "Alps5": base_path / "ViHoP/Alps5/outs/filtered_feature_bc_matrix",
    "Alps_Amir": base_path / "ViHoP/Alps_Amir/outs/filtered_feature_bc_matrix",
}

annotation_file = base_path / "ViHoP/blast_summary.Class.csv"
taxonomy_class = "Dinophyceae"  # ViHoP Class column (dinoflagellates)
homogeneity_threshold = 1.0  # percent is a 0–1 fraction in ViHoP CSVs

output_h5ad = base_path / "Alps_Dinoflagellata.h5ad"

### Load and merge Cell Ranger matrices

In [4]:
adata = merge_cellranger_samples({k: str(v) for k, v in cellranger_paths.items()})
adata

/home/labs/vardi/amirz/pairing/Alps_All_MetaG/ViHoP/Alps2/outs/filtered_feature_bc_matrix
/home/labs/vardi/amirz/pairing/Alps_All_MetaG/ViHoP/Alps5/outs/filtered_feature_bc_matrix
/home/labs/vardi/amirz/pairing/Alps_All_MetaG/ViHoP/Alps_Amir/outs/filtered_feature_bc_matrix


/home/labs/vardi/amirz/.conda/envs/py38/lib/python3.8/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 7127 × 682248
    obs: 'sample'
    layers: 'counts'

### Filter low-count genes and empty cells

In [5]:
sc.pp.filter_genes(adata, min_counts=2)
sc.pp.filter_cells(adata, min_counts=1)
adata

AnnData object with n_obs × n_vars = 7127 × 70282
    obs: 'sample', 'n_counts'
    var: 'n_counts'
    layers: 'counts'

### Match cells to ViHoP taxonomy and retain homogeneous dinoflagellates

In [6]:
total_cells = list(adata.obs_names)
subset_annotation, filtered_annotation = filter_homogeneous_cells(
    total_cells,
    str(annotation_file),
    threshold=homogeneity_threshold,
)
filtered_annotation["Class"].value_counts()

Class
Chrysophyceae          1407
Chlorophyceae           603
Dinophyceae             422
Bacillariophyceae       308
Zygnemophyceae          171
Mediophyceae            171
Rotifera                 73
Trebouxiophyceae         66
Chytridiomycota          55
Kathablepharidea         41
Opalozoa                 37
Cryptophyceae            22
Prymnesiophyceae          9
Pavlovophyceae            8
Choanoflagellatea         5
Embryophyceae             4
CONTH_5                   3
CONThreeP                 3
Cryptophyceae:nucl        2
Hyphochytriomyceta        1
Arthropoda                1
Gyrista_X                 1
Ascomycota                1
Telonemia_XX              1
Mamiellophyceae           1
CONTH_4                   1
Peronosporomycetes        1
Filosa-Thecofilosea       1
Name: count, dtype: int64

In [7]:
adata_sub = subset_by_taxa(
    adata,
    subset_annotation,
    filtered_annotation,
    total_cells,
    level="Class",
    taxon=taxonomy_class,
)
adata_sub

5653 cells with any annotation
1474 cells without annotation
3419 homogeneous cells
422 cells of Dinophyceae


AnnData object with n_obs × n_vars = 422 × 8272
    obs: 'sample', 'n_counts'
    var: 'n_counts'
    layers: 'counts'

### Save raw subset

In [9]:
output_h5ad.parent.mkdir(parents=True, exist_ok=True)
adata_sub.write(output_h5ad)
print(f"Saved {adata_sub.n_obs} cells × {adata_sub.n_vars} genes to {output_h5ad}")

Saved 422 cells × 8272 genes to /home/labs/vardi/amirz/pairing/Alps_All_MetaG/Alps_Dinoflagellata.h5ad
